In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
df = pd.read_csv(r"C:\Users\Maitreya\Desktop\projects\Datasets\human_cognitive_performance.csv")

In [4]:
df.head()

,User_ID,Age,Gender,Sleep_Duration,Stress_Level,Diet_Type,Daily_Screen_Time,Exercise_Frequency,Caffeine_Intake,Reaction_Time,Memory_Test_Score,Cognitive_Score,AI_Predicted_Score
0,U1,57,Female,6.5,3,Non-Vegetarian,6.5,Medium,41,583.33,65,36.71,39.77
1,U2,39,Female,7.6,9,Non-Vegetarian,10.8,High,214,368.24,58,54.35,57.68
2,U3,26,Male,8.2,6,Vegetarian,5.7,Low,429,445.21,49,32.57,29.54
3,U4,32,Male,7.8,9,Vegetarian,8.3,Low,464,286.33,94,70.15,74.59
4,U5,50,Male,9.7,2,Non-Vegetarian,11.3,Medium,365,237.65,62,87.54,91.78


In [5]:
df_new= df.drop(['AI_Predicted_Score','User_ID'],axis=1)

In [6]:
df_new.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 80000 entries, 0 to 79999
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Age                 80000 non-null  int64  
 1   Gender              80000 non-null  object 
 2   Sleep_Duration      80000 non-null  float64
 3   Stress_Level        80000 non-null  int64  
 4   Diet_Type           80000 non-null  object 
 5   Daily_Screen_Time   80000 non-null  float64
 6   Exercise_Frequency  80000 non-null  object 
 7   Caffeine_Intake     80000 non-null  int64  
 8   Reaction_Time       80000 non-null  float64
 9   Memory_Test_Score   80000 non-null  int64  
 10  Cognitive_Score     80000 non-null  float64
dtypes: float64(4), int64(4), object(3)
memory usage: 6.7+ MB


In [7]:
from sklearn.linear_model import LinearRegression

In [8]:
X= df_new.drop('Cognitive_Score',axis=1)
y = df_new['Cognitive_Score']


In [9]:
nominal_cols = ['Gender','Diet_Type']
ordinal_cols = ['Exercise_Frequency']

ordinal_order =[['Low','Medium','High']]

In [10]:
from sklearn.preprocessing import OneHotEncoder,OrdinalEncoder
from sklearn.compose import ColumnTransformer

In [11]:
preprocessor = ColumnTransformer(transformers=[('ord',OrdinalEncoder(categories=ordinal_order),ordinal_cols),
                                 ('nom',OneHotEncoder(drop='first'),nominal_cols)],remainder='passthrough')

In [12]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [13]:
print(df_new["Exercise_Frequency"].unique())
print(df_new["Diet_Type"].unique())
print(df_new["Gender"].unique())

['Medium' 'High' 'Low']
['Non-Vegetarian' 'Vegetarian' 'Vegan']
['Female' 'Male' 'Other']


In [14]:
from sklearn.pipeline import Pipeline
model = Pipeline([
    ("prep", preprocessor),
    ("lr", LinearRegression())
])

In [15]:
model.fit(X_train,y_train)

c:\ProgramData\anaconda3\Lib\site-packages\sklearn\compose\_column_transformer.py:1623: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


Pipeline(steps=[('prep',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('ord',
                                                  OrdinalEncoder(categories=[['Low',
                                                                              'Medium',
                                                                              'High']]),
                                                  ['Exercise_Frequency']),
                                                 ('nom',
                                                  OneHotEncoder(drop='first'),
                                                  ['Gender', 'Diet_Type'])])),
                ('lr', LinearRegression())])

In [16]:
model.score(X_train,y_train)

0.993050009847782

In [17]:
model.score(X_test,y_test)

0.9928571097245852

In [18]:
import pickle


# Save
with open(r"C:/Users/Maitreya/Desktop/cognitive_score_model.pkl", 'wb') as f:
    pickle.dump(model, f)

# Load
with open(r"C:/Users/Maitreya/Desktop/cognitive_score_model.pkl", 'rb') as f:
    model = pickle.load(f)

In [41]:
print("AI_Predicted_Score" in X.columns)
print("AI_Predicted_Score" in X_train.columns)

False
False


In [42]:
df_new.duplicated().sum()

0

In [19]:
from sklearn.metrics import mean_squared_error
import numpy as np

y_pred = model.predict(X_test)

rmse = mean_squared_error(y_test, y_pred, squared=False)
target_std = np.std(y_test)

print("RMSE:", rmse)
print("Target std:", target_std)
print("RMSE / std:", rmse/target_std)

c:\ProgramData\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


RMSE: 1.9402164601207041
Target std: 22.95689750572481
RMSE / std: 0.08451562148747967


In [45]:
from sklearn.inspection import permutation_importance

r = permutation_importance(model, X_test, y_test, n_repeats=5, random_state=42)

import pandas as pd
imp = pd.Series(r.importances_mean, index=X.columns).sort_values(ascending=False)
print(imp.head(10))

Reaction_Time         1.353229e+00
Memory_Test_Score     2.691036e-01
Exercise_Frequency    1.227200e-01
Stress_Level          1.149412e-01
Daily_Screen_Time     8.085963e-02
Sleep_Duration        4.186335e-02
Caffeine_Intake       2.897559e-02
Diet_Type             1.181326e-06
Gender                2.869352e-08
Age                  -1.685521e-07
dtype: float64
